# PDD 기반 민감도 분석 (범퍼 SEA)

핵심 함수(basis, PDD, get_sobol2, find_optimal_degree 등)의 정확성은 `PDD_Legendre_ver4.ipynb`(Ishigami 벤치마크 검증)에서 확인된 상태.

**역할 갱신(지도교수 지침, 2026-08-27, GUIDELINE.md 참고)**: 범퍼 충돌 응답은 비선형·불연속적이라 PDD를 최종 서로게이트로 쓰지 않고, **PDD는 민감도 분석(변수 스크리닝)까지만** 담당. 실제 강건 목적함수·최적화는 크리깅 서로게이트로 진행 예정(별도 노트북, 아직 미작성). 이 노트북에서 하는 일:

1. 데이터 로딩 및 PDD 적합 (민감도 분석용 + 나중에 크리깅과 정확도 비교할 기준선)
2. 변수 스크리닝 (Sobol) — `DESIGN_IDX`/`NOISE_IDX` 결정, 이 결과를 크리깅 노트북에서도 그대로 사용

아래 3~5단계(강건 목적함수·최적화·검증)는 PDD 서로게이트를 직접 쓰던 옛 계획의 흔적으로, 지우지 않고 참고용으로 남겨둠 — 크리깅 노트북 만들 때 구조(설계값 고정 + noise 변수 재샘플링) 그대로 재사용할 것.

In [ ]:
import numpy as np
import pandas as pd

# 검증된 핵심 함수 (PDD_Legendre_ver4.ipynb에서 그대로 가져옴, 수정 없음)

def theoretical_scale(x, domain_min, domain_max, target_min=-1, target_max=1):
    x_std = (x - domain_min) / (domain_max - domain_min)
    scaled_x = x_std * (target_max - target_min) + target_min
    return scaled_x

def min_max_scale(x, min_val=-1, max_val=1):
    # 표본의 경험적 min/max로 스케일링. 정의역을 알고 있으면 theoretical_scale을 쓸 것.
    x_min = x.min(axis=1, keepdims=True)
    x_max = x.max(axis=1, keepdims=True)

    range_mask = (x_max - x_min) == 0
    x_max[range_mask] += 1e-8

    x_std = (x - x_min) / (x_max - x_min)
    scaled_x = x_std * (max_val - min_val) + min_val
    return scaled_x

def basis(x, a):
    if a == 0:
        return np.ones_like(x)
    if a == 1:
        return x

    p_prev2 = np.ones_like(x)
    p_prev1 = x.copy()
    p_n = None

    for n in range(1, a):
        p_n = ((2 * n + 1) * x * p_prev1 - n * p_prev2) / (n + 1)
        p_prev2 = p_prev1
        p_prev1 = p_n

    return p_n

def PDD(x, n, y):
    dim = x.shape[0]
    N = x.shape[1]

    phi = []
    mapping_list = []

    phi.append(np.ones(N))
    mapping_list.append([0] * dim)

    for i in range(dim):
        for j in range(1, n + 1):
            phi.append(basis(x[i, :], j))
            mapping = [0] * dim
            mapping[i] = 1
            mapping_list.append(mapping)

    if y >= 2:
        for i in range(2, n + 1):
            for j in range(1, i):
                for k in range(dim):
                    for l in range(k + 1, dim):
                        x_a = basis(x[k, :], j) * basis(x[l, :], i - j)
                        phi.append(x_a)
                        mapping = [0] * dim
                        mapping[k] = 1
                        mapping[l] = 1
                        mapping_list.append(mapping)

    if y >= 3:
        for i in range(3, n + 1):
            for j in range(1, i - 1):
                for k in range(1, i - j):
                    m = i - j - k
                    for v1 in range(dim):
                        for v2 in range(v1 + 1, dim):
                            for v3 in range(v2 + 1, dim):
                                x_abc = basis(x[v1, :], j) * basis(x[v2, :], k) * basis(x[v3, :], m)
                                phi.append(x_abc)

                                mapping = [0] * dim
                                mapping[v1] = 1
                                mapping[v2] = 1
                                mapping[v3] = 1
                                mapping_list.append(mapping)

    return np.array(phi).T, np.array(mapping_list).T

def calculate_basis_num(N, m, S):
    # PDD 항(계수) 개수를 미리 계산 (공유용 REDLAB_UQ.py의 calculate_basis_num 참고).
    # DOE 샘플 수 계획에 사용: 필요 샘플 수 ~ 이 값의 2~3배.
    # N: 변수 개수, m: 최대 차수, S: 상호작용 제한 최대 차수
    import math
    basis_count = 1
    for s in range(1, S + 1):
        basis_count += math.comb(N, s) * math.comb(m, s)
    return basis_count

def get_sobol2(Ci, mapping, exp_input):
    sensitivity_dict = {}
    total_var_sum = 0.0

    for i in range(1, Ci.shape[0]):
        active_indices = np.where(mapping[:, i] == 1)[0]
        if len(active_indices) == 0:
            continue

        key = "S" + "".join(map(str, sorted(active_indices + 1)))
        contribution = np.var(Ci[i] * exp_input[:, i])

        sensitivity_dict[key] = sensitivity_dict.get(key, 0) + contribution
        total_var_sum += contribution

    for key in sensitivity_dict:
        if total_var_sum > 0:
            sensitivity_dict[key] /= total_var_sum
        else:
            sensitivity_dict[key] = 0

    return sensitivity_dict

def display_detailed_sensitivity(sensitivity_dict):
    results = [{"Interaction": key, "Index": val} for key, val in sensitivity_dict.items()]

    if not results:
        print("출력할 민감도 지수 데이터가 없음.")
        return

    df = pd.DataFrame(results)
    df = df.sort_values(by="Index", ascending=False).reset_index(drop=True)
    df["Rank"] = df["Index"].rank(ascending=False, method="min").astype(int)

    print("\n" + "=" * 45)
    print(f"{'Interaction':^15} | {'Sensitivity Index':^18} | {'Rank':^8}")
    print("-" * 45)
    for _, row in df.iterrows():
        print(f"{row['Interaction']:^15} | {row['Index']:^18.6f} | {row['Rank']:^8}")
    print("-" * 45)
    total_sum = df["Index"].sum()
    print(f"{'Total Sum':^15} | {total_sum:^18.6f} |")
    print("=" * 45)

    return df

def find_optimal_degree(input_data, output_data, max_n=20, max_y=3,
                         val_ratio=0.2, patience=3, tol=1e-4, seed=0, verbose=True):
    # 학습/검증 분리 기반 차수 선택. 학습셋 R^2만으로 고르면 과적합됨(PDD_Legendre_ver4.ipynb 참고).
    rng = np.random.default_rng(seed)
    N_samples = input_data.shape[1]
    perm = rng.permutation(N_samples)
    n_val = max(int(N_samples * val_ratio), 1)
    val_idx, train_idx = perm[:n_val], perm[n_val:]

    X_train, X_val = input_data[:, train_idx], input_data[:, val_idx]
    Y_train, Y_val = output_data[train_idx], output_data[val_idx]

    best_n, best_y = 1, 1
    best_val_r2 = -float("inf")
    no_improve = 0

    for n in range(1, max_n + 1):
        for y in range(1, max_y + 1):
            exp_train, mapping = PDD(X_train, n, y)
            k = exp_train.shape[1]

            if k >= X_train.shape[1] - 1:
                continue

            exp_li = np.linalg.pinv(exp_train)
            Ci = exp_li @ Y_train

            exp_val, _ = PDD(X_val, n, y)
            predicted_val = exp_val @ Ci

            ss_res = np.sum((Y_val - predicted_val) ** 2)
            ss_tot = np.sum((Y_val - np.mean(Y_val)) ** 2)
            val_r2 = 1 - (ss_res / ss_tot)

            if val_r2 > best_val_r2 + tol:
                best_val_r2 = val_r2
                best_n, best_y = n, y
                no_improve = 0
            else:
                no_improve += 1

            if no_improve >= patience:
                if verbose:
                    print(f"검증 R^2가 {patience}회 연속 유의미하게 개선되지 않아 조기 종료.")
                    print(f"선택된 조합: n={best_n}, y={best_y} (검증 R^2: {best_val_r2:.6f})")
                return best_n, best_y

    if verbose:
        print(f"선택된 조합: n={best_n}, y={best_y} (검증 R^2: {best_val_r2:.6f})")

    return best_n, best_y

## 1단계 — 데이터 로딩 및 서로게이트 구축

In [ ]:
# TODO: 실제 DOE 설계점 + FE 해석 결과(SEA)로 교체할 것.
#
# 파일 형식 예시(CSV, 샘플이 행 방향인 경우):
#   design_points.csv : N행 x dim열 (각 행 = 설계변수 값 하나)
#   sea_results.csv   : N행 (각 행 = 그 설계점의 SEA 값)
#
# X_raw = pd.read_csv('design_points.csv').to_numpy()   # (N, dim)
# X = X_raw.T                                            # PDD()는 (dim, N) 형태를 요구
# Y = pd.read_csv('sea_results.csv').to_numpy().ravel()  # (N,)

X = None  # TODO
Y = None  # TODO

# TODO: 각 설계변수의 실제 정의역(카티아 파라미터 LB/UB)을 (dim, 1) 형태로 지정.
# 예: 변수 3개, 두께 1.5~3.0mm / 높이 80~120mm / 폭 60~90mm 라면
# domain_min = np.array([[1.5], [80], [60]])
# domain_max = np.array([[3.0], [120], [90]])
domain_min = None  # TODO
domain_max = None  # TODO

In [ ]:
# X, domain_min, domain_max가 채워지면 아래 실행

X_scaled = theoretical_scale(X, domain_min=domain_min, domain_max=domain_max)

opt_n, opt_y = find_optimal_degree(X_scaled, Y)

exp_input, mapping = PDD(X_scaled, opt_n, opt_y)
Ci = np.linalg.pinv(exp_input) @ Y

print(f"PDD 계수 개수: {Ci.shape[0]}")

## 2단계 — 변수 스크리닝

민감도 지수로 SEA에 영향이 큰 변수/상호작용만 추려서, 이후 최적화 대상 변수 수를 줄임.

In [ ]:
sensitivity = get_sobol2(Ci, mapping, exp_input)
sensitivity_df = display_detailed_sensitivity(sensitivity)

# TODO: 기여도 낮은 변수는 아래 DESIGN_IDX에서 제외하고 명목값(중간값 등)으로 고정.

## 3단계 — 강건 목적함수

설계변수(design)와 불확실 변수(noise, 예: 재료 물성 편차·충돌 조건)를 구분해야 함.
`DESIGN_IDX`는 최적화 대상(카티아에서 직접 조절하는 변수), `NOISE_IDX`는 불확실성을 반영할 변수의 열 인덱스(0-based, 스케일링된 X_scaled 기준).

한 후보 설계 `d`(DESIGN_IDX 값들)에 대해, NOISE_IDX 변수들을 그 범위 안에서 여러 번 뽑아 PDD 서로게이트로 SEA를 다시 계산 → 그 분포의 mean/std를 구함. FE를 다시 돌리는 게 아니라 이미 적합된 다항식만 재평가하는 거라 비용이 거의 없음(강습회 Task2_RDO와 같은 구조, 실제 모델 대신 PDD 서로게이트를 씀).

In [ ]:
DESIGN_IDX = []  # TODO: 최적화 대상 변수의 열 인덱스 (예: [0, 1, 2])
NOISE_IDX = []   # TODO: 불확실성 변수의 열 인덱스 (예: [3, 4])

def estimate_robust_sea(d, opt_n, opt_y, Ci, design_idx=DESIGN_IDX, noise_idx=NOISE_IDX,
                         n_mc=2000, seed=0):
    # d: design_idx에 대응하는 값들 (스케일링된 [-1,1] 공간 기준)
    dim = len(design_idx) + len(noise_idx)
    rng = np.random.default_rng(seed)

    x_full = np.zeros((dim, n_mc))
    x_full[design_idx, :] = np.array(d).reshape(-1, 1)
    x_full[noise_idx, :] = rng.uniform(-1, 1, size=(len(noise_idx), n_mc))

    exp_full, _ = PDD(x_full, opt_n, opt_y)
    sea_samples = exp_full @ Ci

    return sea_samples.mean(), sea_samples.std()

def robust_objective(d, kappa, opt_n, opt_y, Ci):
    mean_sea, std_sea = estimate_robust_sea(d, opt_n, opt_y, Ci)
    return -(mean_sea - kappa * std_sea)  # SEA 최대화 -> 최소화 문제로 부호 반전

## 4단계 — 최적화

In [ ]:
from scipy.optimize import minimize

kappa = 1.0  # TODO: 강건성 가중치, 0~3 정도로 스윕하며 트레이드오프 확인

d0 = np.zeros(len(DESIGN_IDX))  # 초기값 (스케일링 공간의 중앙값)
bounds = [(-1, 1)] * len(DESIGN_IDX)  # 스케일링 공간 기준 box 제약

# TODO: 질량 상한, 최대 침입량/PCF 등 제약이 있으면 scipy 제약 형식으로 추가
result = minimize(robust_objective, d0, args=(kappa, opt_n, opt_y, Ci),
                   method="SLSQP", bounds=bounds)

print(result)

## 5단계 — 검증

- 최적점(`result.x`)을 `theoretical_scale`의 역변환으로 실제 물리 단위로 되돌린 뒤, 실제 카티아+FE로 재해석
- 서로게이트 예측(mean_sea)과 실제 해석 결과 오차 확인
- 오차가 크면 그 근처에 DOE 점 추가 후 1단계부터 재적합